# 01 · Dataset exploration — SROIE & CORD

Goal: understand the receipt corpora before building extraction baselines.

- **SROIE** (`arvindrajan92/sroie_document_understanding`): OCR text with labelled lines — `company`, `date`, `address`, `total`.
- **CORD v2** (`naver-clova-ix/cord-v2`): menu items + total (Indonesian receipts), used as a distribution-shift stress set.

Datasets are **streamed** from Hugging Face so nothing large is written to disk.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "ai-service"))
sys.path.insert(0, str(ROOT / "evals"))
FIG = ROOT / "notebooks" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
print("repo root:", ROOT)


repo root: /Users/zaid/expenseiq


In [2]:
from app.data import load_sroie, load_cord, parse_amount
sroie = load_sroie(120)
cord = load_cord(80)
print('SROIE records:', len(sroie))
print('CORD  records:', len(cord))

SROIE records: 120
CORD  records: 80


## Field coverage (how often each gold field is present)

In [3]:
import pandas as pd
def coverage(records, fields=('merchant','transaction_date','total_amount')):
    return {f: sum(1 for r in records if r.fields.get(f) not in (None,'',[])) for f in fields}
cov = pd.DataFrame({'SROIE': coverage(sroie), 'CORD': coverage(cord)})
cov['SROIE'] = (cov['SROIE']/len(sroie)*100).round(1)
cov['CORD'] = (cov['CORD']/len(cord)*100).round(1)
cov

,SROIE,CORD
merchant,100.0,0.0
transaction_date,100.0,0.0
total_amount,100.0,98.8


## Text length distribution

In [4]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7,4))
ax.hist([len(r.text) for r in sroie], bins=20, alpha=.7, label='SROIE')
ax.hist([len(r.text) for r in cord], bins=20, alpha=.7, label='CORD')
ax.set_xlabel('characters of OCR text'); ax.set_ylabel('# receipts'); ax.legend()
ax.set_title('Receipt text length')
fig.tight_layout(); fig.savefig(FIG/'text_length.png', dpi=120); plt.show()
print('saved', FIG/'text_length.png')

Matplotlib is building the font cache; this may take a moment.


saved /Users/zaid/expenseiq/notebooks/figures/text_length.png


/var/folders/n0/vd3cdjmj6b5895m4qc0392d40000gn/T/ipykernel_5401/1290506829.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FIG/'text_length.png', dpi=120); plt.show()


## Sample receipts

In [5]:
for r in sroie[:2] + cord[:1]:
    print('='*70)
    print(r.source, r.id, r.fields)
    print(r.text[:400])

sroie sroie-0 {'merchant': 'BOOK TA .K(TAMAN DAYA) SDN BHD', 'address': '81100 JOHOR BAHRU,', 'transaction_date': '25/12/2018 8:13:39 PM', 'total_amount': 9.0}
TAN WOON YANN
BOOK TA .K(TAMAN DAYA) SDN BHD
789417-W
NO.53 55,57 & 59, JALAN SAGU 18,
TAMAN DAYA,
81100 JOHOR BAHRU,
JOHOR.
DOCUMENT NO : TD01167104
DATE:
25/12/2018 8:13:39 PM
CASHIER:
MANIS
MEMBER:
CASH BILL
CODE/DESC
PRICE
DISC
AMOUNT
QTY
RM
RM
9556939040116
KF MODELLING CLAY KIDDY FISH
1 PC
*
9.000
0.00
9.00
TOTAL:
ROUR DING ADJUSTMENT:
0.00
ROUND D TOTAL (RM):
9.00
CASH
10.00
CHANGE
1.00
GOO
sroie sroie-1 {'merchant': 'INDAH GIFT & HOME DECO', 'address': '81100 JOHOR BAHRU,JOHOR.', 'transaction_date': '19/10/2018 20:49:59 #01', 'total_amount': 60.3}
TAN WOON YANN
INDAH GIFT & HOME DECO
27,JALAN DEDAP 13,
TAMAN JOHOR JAYA,
81100 JOHOR BAHRU,JOHOR.
TEL:07-3507405
FAX:07-3558160
RECEIPT
19/10/2018 20:49:59 #01
CASHIER: CN
LOCATION/SP: 05 /0531
MB: MO26588
ROOM NO: 01
050100035279
DESC/ITEM
QTY
PRICE
AMT(RM)
ST-PRIVILEGE CARD/

## Takeaways

1. SROIE gives clean `company`/`date`/`total` labels — our primary accuracy signal.
2. CORD is a harder, foreign-currency set with mostly a `total` label — good for robustness checks, not headline accuracy.
3. Receipts are short (hundreds of chars), so `max-model-len` of 8k is ample.